### F1 score and person correction efficient with automatic threshold

In [3]:
import cv2
import numpy as np
from sklearn.metrics import f1_score
import os

def calculate_metrics(test_path, gt_path):
    """Calculate F1 score and Pearson correlation coefficient for a single pair of images"""
    # Read grayscale images (supports 8-bit or 16-bit)
    test_image = cv2.imread(test_path, cv2.IMREAD_GRAYSCALE | cv2.IMREAD_ANYDEPTH)
    gt_image = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE | cv2.IMREAD_ANYDEPTH)
    
    # Check if images were successfully read
    if test_image is None or gt_image is None:
        print(f"Warning: Unable to read images {test_path} or {gt_path}, skipping this pair")
        return None, None
    
    # Ensure image dimensions match
    if test_image.shape != gt_image.shape:
        print(f"Warning: Dimensions of {test_path} and {gt_path} do not match, skipping this pair")
        return None, None
    
    # Binarize images using Otsu's method
    _, test_binary = cv2.threshold(test_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, gt_binary = cv2.threshold(gt_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Convert to 1D arrays and normalize to 0-1 range
    test_flat = (test_binary.flatten() / 255).astype(int)
    gt_flat = (gt_binary.flatten() / 255).astype(int)
    
    # Calculate F1 score (handle zero-division cases)
    f1 = f1_score(gt_flat, test_flat, zero_division=0)
    
    # Calculate Pearson correlation coefficient
    r = np.corrcoef(test_flat, gt_flat)[0, 1]
    
    return f1, r

# Define image pairs (each pair contains prefixes of test image and ground truth image)
image_groups = [
    ("Protein 1", "Protein 1_cross"),
    ("Protein 2", "Protein 2_cross"),
    ("Protein 3", "Protein 3_cross"),
    ("Protein 4", "Protein 4_cross"),
    ("Protein 5", "Protein 5_cross"),
    ("Protein 6", "Protein 6_cross"),
    ("Protein 7", "Protein 7_cross"),
    ("Protein 8", "Protein 8_cross"),
    ("Protein 9", "Protein 9_cross")
]

# Path to the folder containing images
folder_path = "D://decoding simulation"

# Iterate through all pairs and calculate metrics
for test_prefix, gt_prefix in image_groups:
    # Construct image paths (assuming images are in tif format)
    test_path = os.path.join(folder_path, f"{test_prefix}.tif")
    gt_path = os.path.join(folder_path, f"{gt_prefix}.tif")
    
    print(f"\nProcessing: {test_prefix} and {gt_prefix}")
    f1, r = calculate_metrics(test_path, gt_path)
    
    if f1 is not None and r is not None:
        print(f"F1 score: {f1:.4f}")
        print(f"Pearson correlation coefficient (R): {r:.4f}")

print("\nAll pairs processed")


Processing: Protein 1 and Protein 1_cross
F1 score: 1.0000
Pearson correlation coefficient (R): 1.0000

Processing: Protein 2 and Protein 2_cross
F1 score: 1.0000
Pearson correlation coefficient (R): 1.0000

Processing: Protein 3 and Protein 3_cross
F1 score: 1.0000
Pearson correlation coefficient (R): 1.0000

Processing: Protein 4 and Protein 4_cross
F1 score: 1.0000
Pearson correlation coefficient (R): 1.0000

Processing: Protein 5 and Protein 5_cross
F1 score: 1.0000
Pearson correlation coefficient (R): 1.0000

Processing: Protein 6 and Protein 6_cross
F1 score: 1.0000
Pearson correlation coefficient (R): 1.0000

Processing: Protein 7 and Protein 7_cross
F1 score: 1.0000
Pearson correlation coefficient (R): 1.0000

Processing: Protein 8 and Protein 8_cross
F1 score: 1.0000
Pearson correlation coefficient (R): 1.0000

Processing: Protein 9 and Protein 9_cross
F1 score: 1.0000
Pearson correlation coefficient (R): 1.0000

All pairs processed


### Calculate SSIM and PSNR

In [7]:
import os
import cv2
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# Set the path to the folder containing images
folder_path = "D://decoding simulation"

# Define paired image names (without extensions)
image_pairs = [
    ("Protein 1", "Protein 1_cross"),
    ("Protein 2", "Protein 2_cross"),
    ("Protein 3", "Protein 3_cross"),
    ("Protein 4", "Protein 4_cross"),
    ("Protein 5", "Protein 5_cross"),
    ("Protein 6", "Protein 6_cross"),
    ("Protein 7", "Protein 7_cross"),
    ("Protein 8", "Protein 8_cross"),
    ("Protein 9", "Protein 9_cross")
]

# Supported image extensions
image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']

# List to store results
results = []

# Iterate through all paired images
for img1_name, img2_name in image_pairs:
    # Find complete paths of images (with extensions)
    img1_path = None
    img2_path = None
    
    # Find the first image
    for ext in image_extensions:
        potential_path = os.path.join(folder_path, f"{img1_name}{ext}")
        if os.path.exists(potential_path):
            img1_path = potential_path
            break
    
    # Find the second image
    for ext in image_extensions:
        potential_path = os.path.join(folder_path, f"{img2_name}{ext}")
        if os.path.exists(potential_path):
            img2_path = potential_path
            break
    
    # Check if images are found
    if not img1_path:
        print(f"Image not found: {img1_name} (all supported extensions tried)")
        continue
    if not img2_path:
        print(f"Image not found: {img2_name} (all supported extensions tried)")
        continue
    
    # Read images (in grayscale mode)
    img1 = cv2.imread(img1_path, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(img2_path, cv2.IMREAD_GRAYSCALE)
    
    # Check if images are read successfully
    if img1 is None:
        print(f"Failed to read image: {img1_path}")
        continue
    if img2 is None:
        print(f"Failed to read image: {img2_path}")
        continue
    
    # Check if image dimensions match
    if img1.shape != img2.shape:
        print(f"Image dimensions do not match: {img1_name} and {img2_name}")
        continue
    
    # Calculate PSNR
    psnr_value = psnr(img1, img2)
    
    # Calculate SSIM (multichannel=False for grayscale images)
    ssim_value = ssim(img1, img2, multichannel=False)
    
    # Store results
    results.append({
        "Pair": f"{img1_name} vs {img2_name}",
        "PSNR": round(psnr_value, 4),
        "SSIM": round(ssim_value, 4)
    })
    
    print(f"Calculated: {img1_name} vs {img2_name} - PSNR: {psnr_value:.4f}, SSIM: {ssim_value:.4f}")

# Output summary results
print("\n===== Calculation Results Summary =====")
for result in results:
    print(f"{result['Pair']}: PSNR = {result['PSNR']}, SSIM = {result['SSIM']}")

# Save results to text file
output_file = os.path.join(folder_path, "psnr_ssim_results.txt")
with open(output_file, "w") as f:
    f.write("Image Pair\tPSNR\tSSIM\n")
    for result in results:
        f.write(f"{result['Pair']}\t{result['PSNR']}\t{result['SSIM']}\n")

print(f"\nResults saved to: {output_file}")
    

Calculated: Protein 1 vs Protein 1_cross - PSNR: 64.2926, SSIM: 0.9970
Calculated: Protein 2 vs Protein 2_cross - PSNR: 59.7297, SSIM: 0.9999
Calculated: Protein 3 vs Protein 3_cross - PSNR: 50.7876, SSIM: 0.9998
Calculated: Protein 4 vs Protein 4_cross - PSNR: 58.4460, SSIM: 1.0000
Calculated: Protein 5 vs Protein 5_cross - PSNR: 55.4124, SSIM: 0.9999
Calculated: Protein 6 vs Protein 6_cross - PSNR: 58.1048, SSIM: 0.9981
Calculated: Protein 7 vs Protein 7_cross - PSNR: 55.7054, SSIM: 1.0000
Calculated: Protein 8 vs Protein 8_cross - PSNR: 58.7356, SSIM: 0.9962
Calculated: Protein 9 vs Protein 9_cross - PSNR: 50.1264, SSIM: 0.9998

===== Calculation Results Summary =====
Protein 1 vs Protein 1_cross: PSNR = 64.2926, SSIM = 0.997
Protein 2 vs Protein 2_cross: PSNR = 59.7297, SSIM = 0.9999
Protein 3 vs Protein 3_cross: PSNR = 50.7876, SSIM = 0.9998
Protein 4 vs Protein 4_cross: PSNR = 58.446, SSIM = 1.0
Protein 5 vs Protein 5_cross: PSNR = 55.4124, SSIM = 0.9999
Protein 6 vs Protein 6_cr